In [1]:
from matplotlib import image
from matplotlib import pyplot
import numpy as np
import cv2
from PIL import Image

In [2]:
def plot_img(img, disp_size=(7, 7)):
    from matplotlib import pyplot
    pyplot.figure(figsize = disp_size, dpi=80)
    display(pyplot.imshow(img,cmap="gray"))

In [1]:
img = cv2.imread("lena.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [ ]:
orig_image = img.copy()

In [ ]:
original_size = img.shape

In [ ]:
im = img.copy()

In [ ]:
plot_img(im)

In [ ]:
# kernel = np.array([[0, -1, 0],
#                    [-1, 5, -1],
import tensorflow as tf
#                    [0, -1, 0]])

In [ ]:
import matplotlib.pyplot as plt
def plot_histogram(img):
    l = np.zeros(256)
    for r in range(0,img.shape[0]):
        for c in range(0,img.shape[1]):
            l[int(img[r][c])] += 1
    plt.hist(l, bins = 255)
    plt.show()

In [ ]:
row, col = img.shape[:2] #why


def df(img):  # to make a histogram (count distribution frequency)
    values = [0] * 256
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            values[img[i,j]] += 1
    return values


def cdf(hist):  # cumulative distribution frequency
    cdf = [0] * len(hist)   #len(hist) is 256
    cdf[0] = hist[0]
    for i in range(1, len(hist)):
        cdf[i]= cdf[i-1]+hist[i]
    # Now we normalize the histogram
    
    cdf = [ele*255/cdf[-1] for ele in cdf]      # What your function h was doing before
    return cdf

def equalize_image(image):
    my_cdf = cdf(df(image))
    # use linear interpolation of cdf to find new pixel values. Scipy alternative exists
    import numpy as np
    image_equalized = np.interp(image, range(0,256), my_cdf)
    cv2.imwrite('new_image.png', image_equalized)
    image_equalized = cv2.imread('new_image.png', 0)
    return image_equalized

eq = equalize_image(img)

In [ ]:
plot_histogram(eq)

In [ ]:
plot_histogram(orig_image)

In [ ]:
plot_img(eq)

In [ ]:
def padding(im, kernel_size, padding_type = 'full'):
    try:
        padding_type = str(padding_type).lower()
    except:
        padding_type = 'full'
    
    img_w, img_h = im.shape[0], im.shape[1]
    krnl_w, krnl_h = kernel_size, kernel_size
    
    if(padding_type == 'full'):
        pad = np.zeros( (img_w + krnl_w - 1, img_h + krnl_h - 1))
        pad[krnl_w//2: img_w + krnl_w//2, krnl_h//2: img_h + krnl_h//2] = im.copy()
        
    if(padding_type == 'same'):
        pad = np.zeros( (img_w + krnl_w//2 , img_h + krnl_h//2 ))

        pad[krnl_w//2: img_w + krnl_w//2, krnl_h//2: img_h + krnl_h//2] = im.copy()
    if(padding_type == 'valid'):
        pad = im.copy()
    return pad

In [ ]:
import random
def sp_noise(image,prob):
    output = np.zeros(image.shape,np.uint8)
    thres = 1 - prob 
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            rdn = random.random()
            if rdn < prob:
                output[i][j] = 0
            elif rdn > thres:
                output[i][j] = 255
            else:
                output[i][j] = image[i][j]
    return output

In [ ]:
import cv2
import numpy as np
import math
import sys
import time
  
  
#Interpolation kernel
def u(s, a):
    if (abs(s) >= 0) & (abs(s) <= 1):
        return (a+2)*(abs(s)**3)-(a+3)*(abs(s)**2)+1
    elif (abs(s) > 1) & (abs(s) <= 2):
        return a*(abs(s)**3)-(5*a)*(abs(s)**2)+(8*a)*abs(s)-4*a
    return 0
  
  
# Padding
def bicubic_padding(img, H, W, C):
    zimg = np.zeros((H+4, W+4, C))
    zimg[2:H+2, 2:W+2, :C] = img
      
    # Pad the first/last two col and row
    zimg[2:H+2, 0:2, :C] = img[:, 0:1, :C]
    zimg[H+2:H+4, 2:W+2, :] = img[H-1:H, :, :]
    zimg[2:H+2, W+2:W+4, :] = img[:, W-1:W, :]
    zimg[0:2, 2:W+2, :C] = img[0:1, :, :C]
      
    # Pad the missing eight points
    zimg[0:2, 0:2, :C] = img[0, 0, :C]
    zimg[H+2:H+4, 0:2, :C] = img[H-1, 0, :C]
    zimg[H+2:H+4, W+2:W+4, :C] = img[H-1, W-1, :C]
    zimg[0:2, W+2:W+4, :C] = img[0, W-1, :C]
    return zimg
  
  
# Bicubic operation
def bicubic(img, ratio = 0.5, a = -1/2, img_name = 'new_image'):
    
    # Get image size
    img = img[..., tf.newaxis]
    H, W, C = img.shape
      
    # Here H = Height, W = weight,
    # C = Number of channels if the 
    # image is coloured.
    img = bicubic_padding(img, H, W, C)
      
    # Create new image
    dH = math.floor(H*ratio)
    dW = math.floor(W*ratio)
  
    # Converting into matrix
    dst = np.zeros((dH, dW, 3))  
    # np.zeroes generates a matrix 
    # consisting only of zeroes
    # Here we initialize our answer 
    # (dst) as zero
  
    h = 1/ratio
  
    print('Start bicubic interpolation')
    print('It will take a little while...')
    inc = 0
    from tqdm import tqdm
    for c in tqdm(range(C)):
        for j in range(dH):
            for i in range(dW):
                
                # Getting the coordinates of the
                # nearby values
                x, y = i * h + 2, j * h + 2
  
                x1 = 1 + x - math.floor(x)
                x2 = x - math.floor(x)
                x3 = math.floor(x) + 1 - x
                x4 = math.floor(x) + 2 - x
  
                y1 = 1 + y - math.floor(y)
                y2 = y - math.floor(y)
                y3 = math.floor(y) + 1 - y
                y4 = math.floor(y) + 2 - y
                  
                # Considering all nearby 16 values
                mat_l = np.matrix([[u(x1, a), u(x2, a), u(x3, a), u(x4, a)]])
                mat_m = np.matrix([[img[int(y-y1), int(x-x1), c],
                                    img[int(y-y2), int(x-x1), c],
                                    img[int(y+y3), int(x-x1), c],
                                    img[int(y+y4), int(x-x1), c]],
                                   [img[int(y-y1), int(x-x2), c],
                                    img[int(y-y2), int(x-x2), c],
                                    img[int(y+y3), int(x-x2), c],
                                    img[int(y+y4), int(x-x2), c]],
                                   [img[int(y-y1), int(x+x3), c],
                                    img[int(y-y2), int(x+x3), c],
                                    img[int(y+y3), int(x+x3), c],
                                    img[int(y+y4), int(x+x3), c]],
                                   [img[int(y-y1), int(x+x4), c],
                                    img[int(y-y2), int(x+x4), c],
                                    img[int(y+y3), int(x+x4), c],
                                    img[int(y+y4), int(x+x4), c]]])
                mat_r = np.matrix(
                    [[u(y1, a)], [u(y2, a)], [u(y3, a)], [u(y4, a)]])
                  
                # Here the dot function is used to get 
                # the dot product of 2 matrices
                dst[j, i, c] = np.dot(np.dot(mat_l, mat_m), mat_r)
  
    # If there is an error message, it
    # directly goes to stderr
    sys.stderr.write('\n')
      
    # Flushing the buffer
    sys.stderr.flush()
    
    cv2.imwrite(img_name+'.png', dst)
    new_image = cv2.imread(img_name+'.png')
    new_image = cv2.cvtColor(new_image, cv2.COLOR_BGR2GRAY)
    return new_image

In [ ]:
plot_img(bicubic(orig_image, 0.5))

In [ ]:
import math
def euclidian_dist(a,b):
    return np.sqrt(((a[0]-b[0])**2)+((a[1]-b[1])**2))


def near_neighbour(X,P):
    i,j = X[0],X[1]
    A = [[i,j],[i,j+1],[i+1,j],[i+1,j+1]]
    dist = [euclidian_dist(A[0],P),euclidian_dist(A[1],P),euclidian_dist(A[2],P),euclidian_dist(A[3],P)]
    minpos = dist.index(min(dist))
    return A[minpos]        

def NN_interpolation(im,scale_factor):
    row, col = im.shape[0], im.shape[1]
    n_row,n_col = int(scale_factor * row),int(scale_factor * col)
    zoom = np.zeros((n_row,n_col))
    for i in range(n_row):
        for j in range(n_col):
            P = [math.floor(float(i)/scale_factor),math.floor(float(j)/scale_factor)]
            X = [int(i) for i in P]
            zoom[i][j] = im[near_neighbour(X,P)[0]][near_neighbour(X,P)[1]]
    return zoom

In [ ]:
nearest_img = NN_interpolation(orig_image, 0.5)
plot_img(nearest_img)

In [ ]:
def apply_2D_filter(im, Filter, img_name = 'new_image', padding_type = 'full'):
    try:
        img_name = str(img_name)
    except:
        img_name = 'new_image'
    
    kernel = Filter.copy()
    img_w, img_h = im.shape[0], im.shape[1]
    krnl_w, krnl_h = kernel.shape[0], kernel.shape[1]

    pad = padding(im,krnl_h,padding_type)
    new_image = np.zeros((img_w, img_h))

    for r in range(0, pad.shape[0] - krnl_w + 1):
        for c in range(0, pad.shape[1] - krnl_h +1):
            matrix = pad[r: r + krnl_w, c: c + krnl_h]
            new_image[ r , c ] = np.sum(np.multiply(matrix, kernel))
    
    cv2.imwrite(img_name+'.png', new_image)
    new_image = cv2.imread(img_name+'.png')
    new_image = cv2.cvtColor(new0_image, cv2.COLOR_BGR2GRAY)
    return new_image

In [ ]:
def median_filter(x):
    val = list(x.flatten())
    val.sort()
    return val[len(val)//2]

def avg_filter(x):
    val = list(x.flatten())
    return sum(val)//len(val)

def max_filter(x):
    val = list(x.flatten())
    return max(val)
def min_filter(x):
    val = list(x.flatten())
    return min(val)

def apply_nonlinear_filter(im, filter_function, kernel_size = 3, img_name='new_image', padding_type = 'full'):
    try:
        img_name = str(img_name)
    except:
        img_name = 'new_image'
    
    kernel_size = fix_kernel_size(kernel_size)
    img_w, img_h = im.shape[0], im.shape[1]
    krnl_w, krnl_h = kernel_size, kernel_size
    
    pad = padding(im, kernel_size, padding_type )

    new_image = np.zeros((img_w, img_h))

    for r in range(0, pad.shape[0] - krnl_w + 1):
        for c in range(0, pad.shape[1] - krnl_h +1):
            matrix = pad[r: r + krnl_w, c: c + krnl_h]
            new_image[ r , c ] = filter_function(matrix)
    
    cv2.imwrite(img_name+'.png', new_image)
    new_image = cv2.imread(img_name+'.png')
    new_image = cv2.cvtColor(new_image, cv2.COLOR_BGR2GRAY)
    
    return new_image
    

In [ ]:
def fix_kernel_size(kernel_size):
    if(kernel_size % 2 == 0):
        kernel_size += 1
    if(kernel_size <= 1):
        kernel_size = 3
    return kernel_size

In [ ]:
def edge_detection_laplace_neg_4(im, kernel_size, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            if( c == (kernel_size // 2)):
                kernel[r].append(1)
            elif( r == (kernel_size // 2)):
                kernel[r].append(1)
            else:
                kernel[r].append(0)
    kernel[(kernel_size//2)][(kernel_size//2)] = -4
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

def edge_detection_laplace_neg_8(im, kernel_size, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            if( c == (kernel_size // 2)):
                kernel[r].append(1)
            elif( r == (kernel_size // 2)):
                kernel[r].append(1)
            else:
                kernel[r].append(0)
    kernel[(kernel_size//2)][(kernel_size//2)] = -8
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)


def edge_detection_laplace_4(im, kernel_size, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            if( c == (kernel_size // 2)):
                kernel[r].append(-1)
            elif( r == (kernel_size // 2)):
                kernel[r].append(-1)
            else:
                kernel[r].append(0)
    kernel[(kernel_size//2)][(kernel_size//2)] = 4
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

def edge_detection_laplace_8(im, kernel_size, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            if( c == (kernel_size // 2)):
                kernel[r].append(-1)
            elif( r == (kernel_size // 2)):
                kernel[r].append(-1)
            else:
                kernel[r].append(0)
    kernel[(kernel_size//2)][(kernel_size//2)] = 8
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

In [ ]:
def unsharp_masking(im, kernal_size, mask_weight = 1, img_name = 'new_image'):
    #it's high boost if we change the mask_weight to be > 1 otherwise it's unsharp_masking
    if(mask_weight <= 0):
        mask_weight = 1
    blured_image = apply_nonlinear_filter(im, avg_filter, kernal_size)
    mask = im - blured_image
    return im + (mask * mask_weight)

In [ ]:
def sharpen(im, kernel_size = 3, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            if( c == (kernel_size // 2)):
                kernel[r].append(-1)
            elif( r == (kernel_size // 2)):
                kernel[r].append(-1)
            else:
                kernel[r].append(0)
    kernel[(kernel_size//2)][(kernel_size//2)] = 5
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

def sobel_top_positive(im, img_name = 'new_image'):
    kernel_size = fix_kernel_size(3)
    kernel = [[1, 2, 1], [0, 0, 0], [-1, -2 , -1]]
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)un
def sobel_left_positive(im, img_name = 'new_image'):
    kernel_size = fix_kernel_size(3)
    kernel = [[1, 0, -1], [2, 0, -2], [1, 0 , 1]]
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

In [ ]:
def smooth(im , kernel_size, img_name = 'new_image'):
    kernel_size = fix_kernel_size(kernel_size)
    kernel = []
    div = 1/int(kernel_size * kernel_size)
    for r in range(0, kernel_size):
        kernel.append([])
        for c in range(0, kernel_size):
            kernel[r].append(1 * div)
    kernel = np.array(kernel)
    return apply_2D_filter(im, kernel, img_name)

In [ ]:
plot_img(orig_image)

In [ ]:
sobel_top = sobel_top_positive(im)
plot_img(sobel_top)

In [ ]:
sobel_left = sobel_left_positive(im)
plot_img(sobel_left)

In [ ]:
kernal_size = 3

In [ ]:
unsharp_image = unsharp_masking(im, kernal_size, 1)
plot_img(unsharp_image)

In [ ]:
imag_median = apply_nonlinear_filter(im, median_filter, kernal_size)
plot_img(imag_median)

In [ ]:
imag_average = apply_nonlinear_filter(im, avg_filter, kernal_size)
plot_img(imag_average)

In [ ]:
imag_max = apply_nonlinear_filter(im, max_filter, kernal_size)
plot_img(imag_max)

In [ ]:
imag_min = apply_nonlinear_filter(im, min_filter, kernal_size)
plot_img(imag_min)

In [ ]:
img_name = 'new_image'
imag = sharpen(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
img_name = 'new_image'
imag = edge_detection_laplace_4(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
img_name = 'new_image'
imag = edge_detection_laplace_8(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
img_name = 'new_image'
imag = edge_detection_laplace_neg_4(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
img_name = 'new_image'
imag = edge_detection_laplace_neg_8(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
img_name = 'new_image'
imag = smooth(im, kernal_size, img_name)
plot_img(imag)

In [ ]:
new = cv2.imread(img_name+".png",0)

In [ ]:
habd = new - orig_image

In [ ]:
plot_img(habd)

In [ ]:
# kernel = np.array([[0, -1, 0],
#                    [-1, 5, -1],
#                    [0, -1, 0]])
# cv_image_sharp = cv2.filter2D(src=im, ddepth=0, kernel=kernel)

In [ ]:
# plot_img(cv_image_sharp)

# Gaussian smoothing filter

In [ ]:
def dnorm(x, mu, sd):
    return 1 / (np.sqrt(2 * np.pi) * sd) * np.e ** (-np.power((x - mu) / sd, 2) / 2)

def gaussian_kernel(size, sigma=1):
    kernel_1D = np.linspace(-(size // 2), size // 2, size)
    for i in range(size):
        kernel_1D[i] = dnorm(kernel_1D[i], 0, sigma)
    kernel_2D = np.outer(kernel_1D.T, kernel_1D.T)

    kernel_2D *= 1.0 / kernel_2D.max()

    
    
    return kernel_2D/(kernel_2D.shape[0]*kernel_2D.shape[1])

def gaussian_blur(image, kernel_size = 3):
    kernel = gaussian_kernel(kernel_size, sigma=math.sqrt(kernel_size))
    return apply_2D_filter(image, kernel)

In [ ]:
plot_img(gaussian_blur(img, 5))

# Gaussian Noise Filter

In [ ]:
def gauss_noisy(image):
    gauss = np.random.normal(0, 1, image.size)
    gauss = gauss.reshape(image.shape[0],image.shape[1]).astype('uint8')
    img_gauss = cv2.add(image,gauss)
    return img_gauss

In [ ]:
out = gauss_noisy(img)
plot_img(out)

# fourier transform

In [ ]:
img = cv2.imread(img_name+".png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
im = cv2.resize(img , ((img.shape[0]+img.shape[1])//2, (img.shape[0]+img.shape[1])//2))

In [ ]:
f = np.fft.fft2(im)
fshift = np.fft.fftshift(f)
magnitude_spectrum = 20*np.log(np.abs(fshift))
pyplot.imshow(magnitude_spectrum)

In [ ]:
def dft_matrix(input_img):
    rows = input_img.shape[0]
    cols = input_img.shape[1]
    t = np.zeros((rows,cols), complex)
    output_img = np.zeros((rows,cols), complex)
    m = np.arange(rows)
    n = np.arange(cols)
    x = m.reshape((rows,1))
    y = n.reshape((cols,1))
    for row in range(0,rows):
        M1 = 1j * np.sin(-2 * np.pi * y * n / cols) + np.cos(-2 * np.pi * y * n / cols)
        t[row] = np.dot(M1, input_img[row])
    for col in range(0,cols):
        M2 = 1j * np.sin(-2 * np.pi * x * m / cols) + np.cos(-2 * np.pi * x * m / cols)
        output_img[:,col] = np.dot(M2, t[:,col])
    return output_img

In [ ]:
dftma_lena50 = dft_matrix(im)
out_dftma = np.log(np.abs(dftma_lena50))
pyplot.imshow(out_dftma)

In [ ]:
fshift = np.fft.ifft2(dftma_lena50)

In [ ]:
new_image = np.array(fshift)
new_image.shape

In [ ]:
original_size

In [ ]:
new_image = cv2.resize(np.float32(new_image) , (original_size[1],original_size[0]))

In [ ]:
plot_img(new_image)

In [ ]:
# pyplot.figure(figsize=(8, 6), dpi=80)
# pyplot.imshow(new_image.astype(np.uint8), cmap='gray')

In [4]:
from tkinter import * 
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import (FigureCanvasTkAgg, NavigationToolbar2Tk)
from tkinter import filedialog as tkd
from PIL import Image, ImageTk, ImageOps
import os
import cv2

In [5]:
def hoffman():
    global filename
    
    img_name = image_name.get()
    if(img_name == ""):
        img_name = "decompressed"
    img = cv2.imread(filename, 0)
    def combine_nodes(nodes):
        pos = 0
        newnode = []
        if len(nodes) > 1:
            nodes.sort()
            nodes[pos].append("1")                       # assigning values 1 and 0
            nodes[pos+1].append("0")
            combined_node1 = (nodes[pos] [0] + nodes[pos+1] [0])
            combined_node2 = (nodes[pos] [1] + nodes[pos+1] [1])  # combining the nodes to generate pathways
            newnode.append(combined_node1)
            newnode.append(combined_node2)
            newnodes=[]
            newnodes.append(newnode)
            newnodes = newnodes + nodes[2:]
            nodes = newnodes
            huffman_tree.append(nodes)
            combine_nodes(nodes)
        return huffman_tree                                     # huffman tree generation

    
    my_string = np.asarray(img,np.uint8)
    shape = my_string.shape
    a = my_string
    print ("Enetered string is:",my_string)
    my_string = str(my_string.tolist())
    
    
    letters = []
    only_letters = []
    for letter in my_string:
        if letter not in letters:
            frequency = my_string.count(letter)             #frequency of each letter repetition
            letters.append(frequency)
            letters.append(letter)
            only_letters.append(letter)

    nodes = []
    while len(letters) > 0:
        nodes.append(letters[0:2])
        letters = letters[2:]                               # sorting according to frequency
    nodes.sort()
    huffman_tree = []
    huffman_tree.append(nodes)                             #Make each unique character as a leaf node
    
    
    newnodes = combine_nodes(nodes)

    huffman_tree.sort(reverse = True)
    print("Huffman tree with merged pathways:")

    checklist = []
    for level in huffman_tree:
        for node in level:
            if node not in checklist:
                checklist.append(node)
            else:
                level.remove(node)
    count = 0
    for level in huffman_tree:
        print("Level", count,":",level)             #print huffman tree
        count+=1
    print()
    
    letter_binary = []
    if len(only_letters) == 1:
        lettercode = [only_letters[0], "0"]
        letter_binary.append(letter_code*len(my_string))
    else:
        for letter in only_letters:
            code =""
            for node in checklist:
                if len (node)>2 and letter in node[1]:           #genrating binary code
                    code = code + node[2]
            lettercode =[letter,code]
            letter_binary.append(lettercode)
    print(letter_binary)
    print("Binary code generated:")
    for letter in letter_binary:
        print(letter[0], letter[1])

    bitstring =""
    for character in my_string:
        for item in letter_binary:
            if character in item:
                bitstring = bitstring + item[1]
    binary ="0b"+bitstring
                                            # binary code generated
        
        
    uncompressed_file_size = len(my_string)*7
    compressed_file_size = len(binary)-2
    print("Your original file size was", uncompressed_file_size,"bits. The compressed size is:",compressed_file_size)
    print("This is a saving of ",uncompressed_file_size-compressed_file_size,"bits")
    output = open("compressed.txt","w+")
    print("Compressed file generated as compressed.txt")
    output = open("compressed.txt","w+")
    
    
    print("Decoding.......")
    output.write(bitstring)

    bitstring = str(binary[2:])
    uncompressed_string =""
    code =""
    for digit in bitstring:
        code = code+digit
        pos=0                                        #iterating and decoding
        for letter in letter_binary:
            if code ==letter[1]:
                uncompressed_string=uncompressed_string+letter_binary[pos] [0]
                code=""
            pos+=1

    print("Your UNCOMPRESSED data is:")

    temp = re.findall(r'\d+', uncompressed_string)
    res = list(map(int, temp))
    res = np.array(res)
    res = res.astype(np.uint8)
    res = np.reshape(res, shape)
    print(res)
    print("Observe the shapes and input and output arrays are matching or not")
    print("Input image dimensions:",shape)
    print("Output image dimensions:",res.shape)
    data = Image.fromarray(res)
    
    #data.save('uncompressed.png')
    data.save(img_name+'.png')
    img_out = cv2.imread(img_name+".png")
    #img_out = cv2.imread("uncompressed.png")
    plot_img(img_out)

In [6]:
import matplotlib.pyplot as plt
def save_histogram(img, name = 'hist'):
    l = np.zeros(256)
    for r in range(0,img.shape[0]):
        for c in range(0,img.shape[1]):
            l[int(img[r][c])] += 1
    plt.hist(l, bins = 255)
    plt.savefig(name+'.jpg')

In [7]:
def plot(disp_size=(7, 7)):
    from matplotlib import pyplot
    pyplot.figure(figsize = disp_size, dpi=80)
    display(pyplot.imshow(img,cmap="gray"))

In [8]:
global filename
def resizeIt():
    global filename
    filename = tkd.askopenfilename()
    basewidth = 300
    img = Image.open(filename)
    #wpercent = (basewidth / float(img.size[0]))
    #hsize = int((float(img.size[1]) * float(wpercent)))
    #img = img.resize((basewidth, hsize), Image.ANTIALIAS)
    img.save('resize.png')
    load_image('resize.png')

In [9]:
def change_img(img):
    basewidth = 300
    wpercent = (basewidth / float(img.shape[0]))
    hsize = int((float(img.shape[1]) * float(wpercent)))
    #img = img.resize((basewidth, hsize), Image.ANTIALIAS)
    cv2.imwrite('output.png',img)
    #load_image('output.png')
    load_updated_image('output.png')

In [10]:
def resize_image(img):
    resized_image= img.resize((300,300), Image.ANTIALIAS)
    new_image= ImageTk.PhotoImage(resized_image)
    return new_image

In [11]:
def run_filter():
    global filename
    if(Smoothing_clicked.get() != "None"):
        kernel_size = int(Kernel_Size_clicked.get())
        pad_type = Padding_Type_clicked.get()
        img_name = image_name.get()
        if(img_name == ""):
            img_name = "new_image"
        if(Smoothing_clicked.get() == "Median"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = apply_nonlinear_filter(new_image, median_filter, kernel_size, padding_type=pad_type, img_name = img_name)
            change_img(new_image)
            print("exited-----")
        if(Smoothing_clicked.get() == "Max"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = apply_nonlinear_filter(new_image, max_filter, kernel_size, padding_type=pad_type, img_name = img_name)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Min"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = apply_nonlinear_filter(new_image, min_filter, kernel_size, padding_type=pad_type, img_name = img_name)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Average"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = apply_nonlinear_filter(new_image, avg_filter, kernel_size, padding_type=pad_type, img_name = img_name)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Gaussian_blur"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = gaussian_blur(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Sharpen"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = sharpen(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Laplac_neg_4"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = edge_detection_laplace_neg_4(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Laplac_neg_8"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = edge_detection_laplace_neg_8(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Laplac_pos_4"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = edge_detection_laplace_4(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Laplac_pos_8"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = edge_detection_laplace_8(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Sobel_left"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = sobel_left_positive(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
        
        if(Smoothing_clicked.get() == "Sobel_top"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = sobel_top_positive(new_image, kernel_size)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "gaussian_noisy"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = gauss_noisy(new_image)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "salt_pepper"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            try:
                prob = np.abs(float(mask_weigh_sp_probability.get()))
            except:
                prob = 0.05
            new_image = sp_noise(new_image, prob)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "equalization_Histo"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            new_image = equalize_image(new_image)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Unsharp Mask"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            
            new_image = unsharp_masking(new_image, kernal_size, mask_weight = 1)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Highboost"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            
            try:
                mask_weight = np.abs(float(mask_weigh_sp_probability.get()))
            except:
                mask_weight = 1
            
            new_image = unsharp_masking(new_image, kernal_size, mask_weight = 1)
            change_img(new_image)
            print("exited-----")
            
        if(Smoothing_clicked.get() == "Bicubic_Interpolation"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            
            try:
                ratio = np.abs(float(mask_weigh_sp_probability.get()))
            except:
                ratio = 0.5
            
            new_image = bicubic(new_image, ratio = ratio)
            change_img(new_image)
            print("exited-----")
        
        if(Smoothing_clicked.get() == "Nearest_Neighbour"):
            print("entered-----")
            
            print('filename ', filename)
            new_image = cv2.imread(filename, 0)
            
            try:
                ratio = np.abs(float(mask_weigh_sp_probability.get()))
            except:
                ratio = 0.5
        
            new_image = NN_interpolation(new_image, scale_factor = ratio)
            change_img(new_image)
            print("exited-----")
            
        cv2.imwrite(img_name+'.png', new_image)
        orig = cv2.imread(filename, 0)
        save_histogram(orig, "orig_hist")
        save_histogram(new_image, "new_hist")

In [12]:
yaxis = 20 # to move everything on the y-axis
xaxis = 20 # to move everything on the x-axis

xaxisLists = 60 # to move everything on the x-axis


def load_updated_image(path):
    global img2
    load = Image.open(path)
    render = resize_image(load)
    
    img2 = Label(win, image=render)
    img2.image = render
    img2.place(x= 400+xaxis , y= 15+yaxis)

def load_image(path):
    load = Image.open(path)
    load = ImageOps.grayscale(load)
    render = resize_image(load)

    img = Label(win, image=render)
    img.image = render
    img.place (x=  0+xaxis  , y= 15+yaxis)
    
    global img2
    
    img2 = Label(win, image=render)
    img2.image = render
    img2.place(x= 400+xaxis , y= 15+yaxis)



win = Tk()
win.title('Image Processing Project')

win.geometry("800x670")


#Create a canvas
canvas= Canvas(win, width= 800, height=670)
canvas.pack()

load_image('resize.png')


label = Label(win, text='- Original -') 
label.place(x=0+xaxis, y=-10+yaxis)


label = Label(win, text='- Smoothing Filters -')
label.place(x=10+xaxisLists+xaxis, y=330+yaxis)

def clicked():
    csv_file_path = 'sample4'
    print(csv_file_path)
    v.set(csv_file_path)
    df = pd.read_csv(csv_file_path) 
    display_graph(df)
    print("done")


SmoothingOptions = [
    
    #the Smoothening FILTERZzZzZz
    "None",
    "Median",
    "Max",
    "Min",
    "Average",
    "Gaussian_blur",
    
    #the sharpening FILTERZzZzZz
    "Sharpen",
    "Laplac_neg_4",
    "Laplac_neg_8",
    "Laplac_pos_4",
    "Laplac_pos_8",
    "Unsharp Mask",
    "Highboost",
    "Sobel_top",
    "Sobel_left",
    
    #the Noise FILTERZzZzZz
    
    "gaussian_noisy",
    "salt_pepper",
    
    #Frequency THINGZzZzZz
    "equalization_Histo",
    "Bicubic_Interpolation",
    "Nearest_Neighbour"
]

# datatype of menu text
Smoothing_clicked = StringVar()



# initial menu text
Smoothing_clicked.set( "None" )
print(Smoothing_clicked.get())

# Create Dropdown menu
drop = OptionMenu( win , Smoothing_clicked , *SmoothingOptions )
drop.place(x=20+xaxisLists+xaxis, y=350+yaxis)




############                Kernel Size                ###################
##########################################################################
label_5 = Label(win, text='- kernel size -')
label_5.place(x=590+xaxisLists+xaxis, y=330+yaxis)
Kernel_Size_Options = [
    "3", "5", "7", "9", "11", "13", 
    "15", "17", "19", "21", "23", "25"
]
Kernel_Size_clicked = StringVar()
Kernel_Size_clicked.set( "3" )
  
# Create Dropdown menu
drop = OptionMenu( win , Kernel_Size_clicked , *Kernel_Size_Options )
drop.place(x=600+xaxisLists+xaxis, y=350+yaxis)
##########################################################################




############                Padding Type              ###################
##########################################################################
label_6 = Label(win, text='- Padding Type -')
label_6.place(x=390+xaxisLists+xaxis, y=330+yaxis)
Padding_Type_Options = ["full", "same", "valid" ]
Padding_Type_clicked = StringVar()
Padding_Type_clicked.set( "full" )
  
# Create Dropdown menu
drop = OptionMenu( win , Padding_Type_clicked , *Padding_Type_Options )
drop.place(x=400+xaxisLists+xaxis, y=350+yaxis)
##########################################################################



############           Mask_Weight/ SP Probability     ###################
##########################################################################
label = Label(win, text = 'mask_weigh || sp_probability')
label.place(x=200+xaxisLists+xaxis, y=330+yaxis)
mask_weigh_sp_probability = Entry(win,textvariable = "name_var", font=('calibre',10,'normal'))
mask_weigh_sp_probability.place(x=210+xaxisLists+xaxis, y=350+yaxis)
##########################################################################


############           Histograms Plotting for img     ###################
##########################################################################
label = Label(win, text = 'saved image name')
label.place(x=220+xaxisLists+xaxis, y=430+yaxis)
image_name = Entry(win,textvariable = "name_var_2", font=('calibre',10,'normal'))
image_name.place(x=210+xaxisLists+xaxis, y=450+yaxis)
##########################################################################




############                Padding Type              ###################
##########################################################################
compress_btn=Button(text='Compress/Decompress', command=hoffman)
compress_btn.place(x=460+xaxis, y=400+yaxis)
##########################################################################





import_btn=Button(text='Import image', command=resizeIt)
import_btn.place(x=310+xaxis, y=100+yaxis)

btn_apply = Button(win, text='Apply', width=10,
             height=1, bd='2',command=run_filter)  
btn_apply.place(x=310+xaxis, y=200+yaxis)

win.mainloop()

FileNotFoundError: [Errno 2] No such file or directory: 'resize.png'